# Backtest Split Gantt Chart

This notebook compares how `track_b_backtest.py` and the CA (`cluster_action_backtest.py`) pipeline assign Track B windows to train, validation, and test.

- CA uses the split dates saved by the experiment pipeline.
- `track_b_backtest.py` recomputes train/validation/test from the full `window_end_dates` list.

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def resolve_track_b_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "results" / "cluster_action_rankings_k4" / "split_dates.csv").exists():
        return cwd
    candidate = cwd / "Final_Project" / "track_b"
    if (candidate / "results" / "cluster_action_rankings_k4" / "split_dates.csv").exists():
        return candidate
    raise FileNotFoundError(
        "Could not find results/cluster_action_rankings_k4/split_dates.csv. "
        "Run this notebook from the repo root or Final_Project/track_b."
    )


TRACK_B_ROOT = resolve_track_b_root()
CA_RESULT_DIR = TRACK_B_ROOT / "results" / "cluster_action_rankings_k4"
SPLIT_DATES_PATH = CA_RESULT_DIR / "split_dates.csv"
OUTPUT_PATH = TRACK_B_ROOT / "results" / "backtest_split_gantt_k4.png"
OVERLAP_OUTPUT_PATH = TRACK_B_ROOT / "results" / "backtest_split_overlap_k4.png"

print(f"Track B root: {TRACK_B_ROOT}")
print(f"CA split dates: {SPLIT_DATES_PATH}")
print(f"Plot output: {OUTPUT_PATH}")
print(f"Overlap plot output: {OVERLAP_OUTPUT_PATH}")

In [ ]:
ca_split_dates = pd.read_csv(SPLIT_DATES_PATH, parse_dates=["Date"])
ca_split_dates = ca_split_dates.loc[:, ["Date", "split"]].copy()
ca_split_dates["method"] = "CA backtest\ncluster_action_backtest.py"

# This mirrors split_ordered_train_random_holdout_indices() in encoder_only_transformer.py,
# as used by track_b_backtest.py when it recomputes splits from window_end_dates.
def split_ordered_train_random_holdout_indices(
    n_obs: int,
    train_ratio: float,
    validation_ratio: float,
    test_ratio: float,
) -> dict[str, np.ndarray]:
    if n_obs < 3:
        raise ValueError(f"Need at least 3 observations, got {n_obs}.")

    total_ratio = train_ratio + validation_ratio + test_ratio
    if total_ratio <= 0:
        raise ValueError("train_ratio + validation_ratio + test_ratio must be positive.")

    normalized_train_ratio = train_ratio / total_ratio
    normalized_validation_ratio = validation_ratio / total_ratio
    normalized_test_ratio = test_ratio / total_ratio

    train_count = max(1, int(n_obs * normalized_train_ratio))
    train_count = min(train_count, n_obs - 2)
    holdout_indices = np.arange(train_count, n_obs, dtype=int)
    if len(holdout_indices) < 2:
        raise ValueError("Need at least 2 holdout observations for validation/test.")

    holdout_total_ratio = normalized_validation_ratio + normalized_test_ratio
    if holdout_total_ratio <= 0:
        raise ValueError("validation_ratio + test_ratio must be positive.")

    validation_share = normalized_validation_ratio / holdout_total_ratio
    validation_count = int(round(len(holdout_indices) * validation_share))
    validation_count = max(1, validation_count)
    validation_count = min(validation_count, len(holdout_indices) - 1)

    return {
        "train": np.arange(train_count, dtype=int),
        "validation": holdout_indices[:validation_count],
        "test": holdout_indices[validation_count:],
    }


all_window_dates = pd.Index(ca_split_dates["Date"].sort_values().unique(), name="Date")
track_b_indices = split_ordered_train_random_holdout_indices(
    n_obs=len(all_window_dates),
    train_ratio=0.70,
    validation_ratio=0.20,
    test_ratio=0.10,
)

track_b_split_dates = pd.concat(
    [
        pd.DataFrame({"Date": all_window_dates.take(indices), "split": split_name})
        for split_name, indices in track_b_indices.items()
    ],
    ignore_index=True,
)
track_b_split_dates["method"] = "Track B backtest\ntrack_b_backtest.py"

plot_dates = pd.concat([track_b_split_dates, ca_split_dates], ignore_index=True)
plot_dates["split"] = pd.Categorical(
    plot_dates["split"],
    categories=["train", "validation", "test"],
    ordered=True,
)

split_summary = (
    plot_dates.groupby(["method", "split"], observed=True)
    .agg(start_date=("Date", "min"), end_date=("Date", "max"), n_windows=("Date", "nunique"))
    .reset_index()
)
split_summary

In [ ]:
overlap = pd.crosstab(
    ca_split_dates.sort_values("Date")["split"].to_numpy(),
    track_b_split_dates.sort_values("Date")["split"].to_numpy(),
    rownames=["CA split"],
    colnames=["Track B recomputed split"],
)
overlap

The table above compares the exact same `window_end_dates` under both assignment rules. Off-diagonal counts are windows that move to a different split when `track_b_backtest.py` recomputes splits.

The blank calendar gaps between CA train/validation/test are expected: the experiment pipeline splits the raw daily rows first, then forms 60-trading-day windows inside each split. The first validation/test window therefore ends about 60 trading days after the raw split boundary.

In [ ]:
assignment_frame = ca_split_dates.loc[:, ["Date", "split"]].rename(columns={"split": "ca_split"})
assignment_frame = assignment_frame.merge(
    track_b_split_dates.loc[:, ["Date", "split"]].rename(columns={"split": "track_b_split"}),
    on="Date",
    how="inner",
).sort_values("Date").reset_index(drop=True)
assignment_frame["same_split"] = assignment_frame["ca_split"] == assignment_frame["track_b_split"]
assignment_frame["transition"] = assignment_frame["ca_split"].astype(str) + " -> " + assignment_frame["track_b_split"].astype(str)

mismatch_summary = (
    assignment_frame.loc[~assignment_frame["same_split"]]
    .groupby(["ca_split", "track_b_split"], sort=False)
    .agg(start_date=("Date", "min"), end_date=("Date", "max"), n_windows=("Date", "nunique"))
    .reset_index()
)
mismatch_summary

In [ ]:
split_order = ["train", "validation", "test"]
colors = {
    "train": "#8aa6c1",
    "validation": "#e2b66f",
    "test": "#75a87f",
}


def contiguous_segments(frame: pd.DataFrame, label_col: str) -> pd.DataFrame:
    ordered = frame.sort_values("Date").reset_index(drop=True).copy()
    group_id = ordered[label_col].ne(ordered[label_col].shift()).cumsum()
    return (
        ordered.groupby(group_id)
        .agg(
            label=(label_col, "first"),
            start_date=("Date", "min"),
            end_date=("Date", "max"),
            n_windows=("Date", "nunique"),
        )
        .reset_index(drop=True)
    )


ca_segments = contiguous_segments(assignment_frame.rename(columns={"ca_split": "split"}), "split")
track_b_segments = contiguous_segments(assignment_frame.rename(columns={"track_b_split": "split"}), "split")

fig, ax = plt.subplots(figsize=(15, 4.8))
bar_height = 0.34

def draw_segments(segments: pd.DataFrame, y: float) -> None:
    for _, row in segments.iterrows():
        start = pd.Timestamp(row["start_date"])
        end = pd.Timestamp(row["end_date"])
        start_num = mdates.date2num(start)
        end_num = mdates.date2num(end)
        width = max(end_num - start_num + 1, 1)
        label = str(row["label"])
        color = colors[label]
        ax.broken_barh(
            [(start_num, width)],
            (y - bar_height / 2, bar_height),
            facecolors=color,
            edgecolors="white",
            linewidth=1.3,
            alpha=0.92,
        )
        text = f"{label}\n{int(row['n_windows'])}"
        ax.text(
            start + (end - start) / 2,
            y,
            text,
            ha="center",
            va="center",
            fontsize=8.6,
            color="#1f2933",
            fontweight="bold",
        )


draw_segments(ca_segments, 1.0)
draw_segments(track_b_segments, 0.0)

# Highlight only the Track B recomputed row. CA itself has clean, non-overlapping splits;
# these marks show dates from CA train/validation that Track B reassigns after recomputing splits.
for i, row in mismatch_summary.iterrows():
    start = pd.Timestamp(row["start_date"])
    end = pd.Timestamp(row["end_date"])
    start_num = mdates.date2num(start)
    end_num = mdates.date2num(end)
    width = max(end_num - start_num + 1, 1)
    mid = start + (end - start) / 2
    label = f"{row['ca_split']} -> {row['track_b_split']}\n{int(row['n_windows'])} windows"

    ax.broken_barh(
        [(start_num, width)],
        (0.0 - bar_height / 2, bar_height),
        facecolors="none",
        edgecolors="#bf4b4b",
        linewidth=1.7,
        hatch="///",
    )
    ax.annotate(
        f"CA {row['ca_split']} dates\nreassigned as Track B {row['track_b_split']}\n{int(row['n_windows'])} windows",
        xy=(mid, 0.0),
        xytext=(mid, 1.58 if i == 0 else 1.36),
        ha="center",
        va="center",
        fontsize=9,
        color="#7f1d1d",
        fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.28", facecolor="#fff5f5", edgecolor="#bf4b4b", linewidth=1.0),
        arrowprops=dict(arrowstyle="->", color="#bf4b4b", linewidth=1.1),
    )

ax.set_yticks([1.0, 0.0])
ax.set_yticklabels([
    "CA split\ncluster_action_backtest.py",
    "Track B recomputed split\ntrack_b_backtest.py",
], fontsize=10)
ax.set_ylim(-0.55, 1.85)
ax.set_title("Window Split Reassignment When Track B Recomputes Splits", fontsize=15, fontweight="bold", pad=14)
ax.set_xlabel("Window end date")
ax.grid(axis="x", color="#d0d7de", linewidth=0.8, alpha=0.7)
ax.set_axisbelow(True)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=3))

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

legend_handles = [
    plt.Rectangle((0, 0), 1, 1, color=colors[split_name], label=split_name)
    for split_name in split_order
]
legend_handles.append(plt.Rectangle((0, 0), 1, 1, facecolor="white", edgecolor="#bf4b4b", hatch="///", label="reassigned dates"))
ax.legend(handles=legend_handles, loc="lower center", bbox_to_anchor=(0.5, -0.28), ncol=4, frameon=False)

fig.tight_layout()
fig.savefig(OVERLAP_OUTPUT_PATH, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved overlap plot to: {OVERLAP_OUTPUT_PATH}")

In [ ]:
method_order = [
    "Track B backtest\ntrack_b_backtest.py",
    "CA backtest\ncluster_action_backtest.py",
]
split_order = ["train", "validation", "test"]
colors = {
    "train": "#8aa6c1",
    "validation": "#e2b66f",
    "test": "#75a87f",
}

fig, ax = plt.subplots(figsize=(15, 4.8))
bar_height = 0.34

for y, method in enumerate(method_order):
    method_rows = split_summary.loc[split_summary["method"] == method].copy()
    for split_name in split_order:
        row = method_rows.loc[method_rows["split"] == split_name]
        if row.empty:
            continue
        row = row.iloc[0]
        start = pd.Timestamp(row["start_date"])
        end = pd.Timestamp(row["end_date"])
        start_num = mdates.date2num(start)
        end_num = mdates.date2num(end)
        width = max(end_num - start_num + 1, 1)

        ax.broken_barh(
            [(start_num, width)],
            (y - bar_height / 2, bar_height),
            facecolors=colors[split_name],
            edgecolors="white",
            linewidth=1.4,
        )

        label = f"{split_name}\n{int(row['n_windows'])} windows"
        ax.text(
            start + (end - start) / 2,
            y,
            label,
            ha="center",
            va="center",
            fontsize=9,
            color="#1f2933",
            fontweight="bold",
        )

        ax.axvline(start, color="white", linewidth=0.8, alpha=0.8)
        ax.axvline(end, color="white", linewidth=0.8, alpha=0.8)

ax.set_yticks(range(len(method_order)))
ax.set_yticklabels(method_order, fontsize=10)
ax.set_ylim(-0.65, len(method_order) - 0.35)
ax.set_title("Track B Window Split Comparison", fontsize=15, fontweight="bold", pad=14)
ax.set_xlabel("Date")
ax.grid(axis="x", color="#d0d7de", linewidth=0.8, alpha=0.7)
ax.set_axisbelow(True)

ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=3))

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

legend_handles = [
    plt.Rectangle((0, 0), 1, 1, color=colors[split_name], label=split_name)
    for split_name in split_order
]
ax.legend(handles=legend_handles, loc="lower center", bbox_to_anchor=(0.5, -0.28), ncol=3, frameon=False)

fig.autofmt_xdate(rotation=0)
fig.tight_layout()
fig.savefig(OUTPUT_PATH, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved plot to: {OUTPUT_PATH}")